# NB2 — Anonymisation Pipeline (Google Colab version)

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phase 4** of the implementation plan. **Google Colab version.**  
Inputs: `COS700/data/analysis.parquet`, `COS700/results/feature_meta.csv`, `COS700/results/gini_importance.csv`  
Outputs: 12 parquet files in `COS700/data/anonymised/`

## Before running
1. Run NB1 (Colab version) first — outputs must be in `COS700/` on your Drive.
2. Set `DRIVE_ROOT` in the setup cell below if your folder path differs.
3. Run all cells top-to-bottom.


In [ ]:
from pathlib import Path
# ── Install dependencies (only needed once per Colab session) ─────────────────
import subprocess, sys
pkgs = ['pyarrow']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Set your Drive root folder ────────────────────────────────────────────────
# Change this if your COS700 folder is in a different location on Drive
DRIVE_ROOT = '/content/drive/MyDrive/COS700'

print(f'Drive mounted. Using root: {DRIVE_ROOT}')


# NB2 — Anonymisation Pipeline

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phase 4** of the implementation plan.  
Inputs: `data/analysis.parquet`, `results/feature_meta.csv`, `results/gini_importance.csv`  
Outputs: 12 parquet files in `data/anonymised/`

| Variant | File |
|---|---|
| Noise α=0.1 | `noise_alpha0.1.parquet` |
| Noise α=0.5 | `noise_alpha0.5.parquet` |
| Suppression 25% | `suppression_0.25.parquet` |
| Suppression 50% | `suppression_0.50.parquet` |
| Suppression 75% | `suppression_0.75.parquet` |
| DP ε=0.01 | `dp_eps0.01.parquet` |
| DP ε=0.1 | `dp_eps0.1.parquet` |
| DP ε=0.5 | `dp_eps0.5.parquet` |
| DP ε=1.0 | `dp_eps1.0.parquet` |
| DP ε=5.0 | `dp_eps5.0.parquet` |
| DP ε=10.0 | `dp_eps10.0.parquet` |
| DP clipping-only (ε→∞) | `dp_clip_only.parquet` |

**Run all cells top-to-bottom. Do not proceed to NB3 until the gate checklist at the bottom passes.**

## 0. Imports & constants

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT          = Path(DRIVE_ROOT)
DATA_DIR      = ROOT / 'data'
ANON_DIR      = DATA_DIR / 'anonymised'
RESULTS_DIR   = ROOT / 'results'
ANON_DIR.mkdir(parents=True, exist_ok=True)

# ── Experiment constants (from revised proposal §5.3 / §5.6) ──────────────────
RANDOM_STATE        = 42
LABEL_COL           = 'Label'
NOISE_ALPHAS        = [0.1, 0.5]
SUPPRESSION_LEVELS  = [0.25, 0.50, 0.75]
DP_EPSILONS         = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

print('Imports OK. ROOT:', ROOT)

## 1. Load inputs

In [ ]:
# ── Analysis set ───────────────────────────────────────────────────────────────
analysis = pd.read_parquet(DATA_DIR / 'analysis.parquet')
feature_cols = [c for c in analysis.columns if c != LABEL_COL]
d = len(feature_cols)
print(f'Analysis set: {len(analysis):,} rows × {d} features + Label')

# ── Feature metadata (bounds for DP) ──────────────────────────────────────────
meta = pd.read_csv(RESULTS_DIR / 'feature_meta.csv')
assert len(meta) == d, f'feature_meta has {len(meta)} rows but d={d}'
bounds = meta.set_index('feature')[['a_j', 'b_j', 'delta_f']].to_dict('index')
print(f'feature_meta loaded: {len(meta)} rows')

# ── Gini importance (for suppression ordering) ─────────────────────────────────
gini = pd.read_csv(RESULTS_DIR / 'gini_importance.csv')
# Ascending importance = least important first (suppressed first)
suppression_order = gini.sort_values('importance', ascending=True)['feature'].tolist()
suppression_order = [f for f in suppression_order if f in feature_cols]
assert len(suppression_order) == d, 'Mismatch between gini order and feature_cols'
print(f'Suppression order loaded: {len(suppression_order)} features')
print(f'First suppressed (lowest importance): {suppression_order[0]}')
print(f'Last  suppressed (highest importance): {suppression_order[-1]}')

## 2. Helper: save & verify

In [ ]:
def save_variant(df: pd.DataFrame, name: str) -> None:
    """Save one anonymised variant to data/anonymised/{name}.parquet."""
    path = ANON_DIR / f'{name}.parquet'
    df.to_parquet(path, engine='pyarrow', index=False)

def verify_variant(df: pd.DataFrame, original: pd.DataFrame, name: str) -> None:
    """Assert shape integrity and no NaN introduced."""
    assert df.shape == original.shape, (
        f'{name}: shape {df.shape} != original {original.shape}'
    )
    assert df.isnull().sum().sum() == 0, f'{name}: NaN values introduced'
    # Label must be unchanged
    assert (df[LABEL_COL] == original[LABEL_COL]).all(), f'{name}: Label column changed'

print('Helper functions defined.')

## 3. Noise perturbation

For each α ∈ {0.1, 0.5}: add `N(0, (α·σ_j)²)` per feature, where `σ_j` is the
standard deviation of feature *j* on the **unmodified** analysis set.

No formal privacy claim. Serves as the informal additive-perturbation baseline.

In [ ]:
# Pre-compute per-feature std from the UNMODIFIED analysis set (once)
feature_stds = analysis[feature_cols].std(axis=0).to_numpy()  # shape (d,)
feature_values = analysis[feature_cols].to_numpy(dtype=float)  # shape (N, d)

print('Noise perturbation:')
for alpha in NOISE_ALPHAS:
    rng = np.random.default_rng(RANDOM_STATE)
    noise = rng.normal(
        loc=0.0,
        scale=(alpha * feature_stds),  # broadcast: (d,) → same scale per column
        size=feature_values.shape
    )
    noisy = feature_values + noise

    out = analysis.copy()
    out[feature_cols] = noisy

    name = f'noise_alpha{alpha}'
    verify_variant(out, analysis, name)
    save_variant(out, name)

    # Spot-check: mean shift should be ≈0, std increase should be ≈α·σ_j
    mean_shift = np.abs(out[feature_cols].mean() - analysis[feature_cols].mean()).mean()
    print(f'  α={alpha}: saved  |  mean abs shift = {mean_shift:.4f}  (expect ≈0)')

print('Done.')

## 4. Suppression (mean-imputation)

For each level ∈ {0.25, 0.50, 0.75}: replace the `floor(d × level)` least-important
features with their mean, computed from the **unmodified** analysis set.

Features are suppressed in ascending Gini-importance order (least important first).

In [ ]:
# Pre-compute unmodified means (once, before any anonymisation)
feature_means = analysis[feature_cols].mean().to_dict()

print("Suppression (mean-imputation):")
for level in SUPPRESSION_LEVELS:
    n_suppress = int(np.floor(d * level))
    suppressed_feats = suppression_order[:n_suppress]  # least important first

    out = analysis.copy()
    for feat in suppressed_feats:
        out[feat] = feature_means[feat]  # replace with unmodified-set mean

    name = f"suppression_{level:.2f}"
    verify_variant(out, analysis, name)
    # Suppressed columns: allow tiny floating-point residual (mean not exactly representable)
    suppressed_stds = out[suppressed_feats].std()
    assert (suppressed_stds < 1e-6).all(), (
        f"{name}: suppressed columns have unexpectedly large std"
    )
    # Unsuppressed columns must be identical to original
    kept_feats = [f for f in feature_cols if f not in suppressed_feats]
    assert (out[kept_feats].values == analysis[kept_feats].values).all(), (
        f"{name}: unsuppressed columns changed"
    )
    save_variant(out, name)

    print(f"  level={level}: suppressed {n_suppress}/{d} features  "
          f"(first={suppressed_feats[0]}, last={suppressed_feats[-1]})")

print("Done.")


## 5. Differential privacy — per-value ε-local DP

For each feature *j*:
1. **Clip** values to `[a_j, b_j]` (data-independent public bounds from `feature_meta.csv`).
2. **Sensitivity** `Δf_j = b_j − a_j` (global sensitivity of the per-value identity release).
3. **Noise**: add `Laplace(0, Δf_j / ε)` per value.

Each per-value release satisfies **ε-local DP**. The full record (d features) satisfies **d·ε-local DP** by sequential composition. Privacy unit is the **flow**, not the user.

Also produces a **clipping-only baseline** (ε → ∞): clip but add no noise. Used in NB3 to separate clipping distortion from noise distortion.

In [ ]:
# Pre-compute per-feature clip bounds and delta_f arrays
a_arr = np.array([bounds[f]['a_j']   for f in feature_cols], dtype=float)  # (d,)
b_arr = np.array([bounds[f]['b_j']   for f in feature_cols], dtype=float)  # (d,)
df_arr = np.array([bounds[f]['delta_f'] for f in feature_cols], dtype=float)  # (d,)

assert (df_arr > 0).all(), 'Some delta_f values are zero — check feature_meta.csv'

X = analysis[feature_cols].to_numpy(dtype=float)  # (N, d) — original values

# Step 1: clip once (same for all ε and the baseline)
X_clipped = np.clip(X, a_arr, b_arr)  # (N, d)

# Clipping report
n_clipped_per_feature = (X != X_clipped).sum(axis=0)  # (d,)
pct_clipped = n_clipped_per_feature / len(X) * 100
total_clipped_values = n_clipped_per_feature.sum()
features_with_clipping = (n_clipped_per_feature > 0).sum()
print(f'Clipping summary:')
print(f'  Features with any clipping:    {features_with_clipping} / {d}')
print(f'  Total clipped values:          {total_clipped_values:,}')
print(f'  Worst clipped features (top 5):')
top5_idx = np.argsort(pct_clipped)[::-1][:5]
for idx in top5_idx:
    if pct_clipped[idx] > 0:
        print(f'    {feature_cols[idx]:<35}  {pct_clipped[idx]:.2f}% clipped')

In [ ]:
# ── Clipping-only baseline (ε → ∞): clip but no noise ─────────────────────────
out_clip = analysis.copy()
out_clip[feature_cols] = X_clipped
name = 'dp_clip_only'
verify_variant(out_clip, analysis, name)
# All values must be in [a_j, b_j]
for feat, a, b in zip(feature_cols, a_arr, b_arr):
    col = out_clip[feat].values
    assert col.min() >= a - 1e-9 and col.max() <= b + 1e-9, \
        f'Clip bounds violated for {feat}: [{col.min()}, {col.max()}] not in [{a}, {b}]'
save_variant(out_clip, name)
print(f'dp_clip_only saved  (ε → ∞ baseline)')

In [ ]:
# ── DP sweep: ε ∈ {0.01, 0.1, 0.5, 1.0, 5.0, 10.0} ──────────────────────────
print('Differential privacy (per-value ε-local DP):')
print(f'  d = {d}, so flow-level guarantee = d·ε per flow')
print()

for eps in DP_EPSILONS:
    rng = np.random.default_rng(RANDOM_STATE)

    # Laplace scale = Δf_j / ε  (broadcast across rows)
    scale = df_arr / eps          # (d,)
    noise = rng.laplace(
        loc=0.0,
        scale=scale,              # per-feature scale
        size=X_clipped.shape      # (N, d)
    )
    X_noised = X_clipped + noise  # (N, d)

    out = analysis.copy()
    out[feature_cols] = X_noised

    name = f'dp_eps{eps}'
    verify_variant(out, analysis, name)
    save_variant(out, name)

    # Report: mean absolute noise per feature at this ε
    mean_abs_noise = np.abs(noise).mean(axis=0).mean()  # scalar
    print(f'  ε={eps:<5}  flow-level={d}·{eps}={d*eps:<8.2f}  '
          f'mean|noise|={mean_abs_noise:.4g}  saved')

print('\nDone.')

## 6. Gate checklist

All assertions must pass before running NB3.

In [ ]:
expected_files = (
    [f'noise_alpha{a}' for a in NOISE_ALPHAS] +
    [f'suppression_{l:.2f}' for l in SUPPRESSION_LEVELS] +
    [f'dp_eps{e}' for e in DP_EPSILONS] +
    ['dp_clip_only']
)

all_passed = True
for name in expected_files:
    path = ANON_DIR / f'{name}.parquet'
    exists = path.exists()
    if exists:
        df_check = pd.read_parquet(path)
        shape_ok = df_check.shape == analysis.shape
        nan_ok   = df_check.isnull().sum().sum() == 0
        ok = exists and shape_ok and nan_ok
    else:
        ok = False
    icon = '✓' if ok else '✗'
    print(f'  {icon}  {name}.parquet')
    if not ok:
        all_passed = False
        if not exists:
            print(f'       → file missing')
        elif not shape_ok:
            print(f'       → shape {df_check.shape} != {analysis.shape}')
        elif not nan_ok:
            print(f'       → contains NaN values')

print()
print(f'Total variants: {len(expected_files)}')
if all_passed:
    print('All gates PASSED. Ready to run NB3.')
else:
    print('Some gates FAILED. Fix the issues above before running NB3.')
    raise AssertionError('Gate check failed — see output above.')